In [1]:
import pandas as pd
import numpy as np

# Tabela1 autorzy
autorzy = pd.DataFrame({
    'autor_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'imie': ['Olga', 'Stanisław', 'Andrzej', 'Wisława', 'Ryszard', 'Dorota', 'Szczepan', 'Jacek'],
    'nazwisko': ['Tokarczuk', 'Lem', 'Sapkowski', 'Szymborska', 'Kapuściński', 'Masłowska', 'Twardoch', 'Dehnel'],
    'kraj': ['Polska'] * 8,
    'nagrody': ['Nobel', 'SFF', 'SFF', 'Nobel', 'Reporter', 'Polityka', 'NIKE', 'NIKE']
})

# Tabela2 książki
ksiazki = pd.DataFrame({
    'ksiazka_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    'autor_id': [1, 1, 2, 2, 3, 3, 4, 5, 6, 7, 7, 8],
    'tytul': ['Księgi Jakubowe', 'Bieguni', 'Solaris', 'Cyberiada', 'Wiedźmin', 'Narrenturm', 'Wiersze wybrane', 'Podróże z Herodotem', 'Wojna polsko-ruska', 'Morfina', 'Król', 'Lala'],
    'kategoria': ['historyczna', 'obyczajowa', 'sci-fi', 'sci-fi', 'fantasy', 'fantasy', 'poezja', 'reportaz', 'obyczajowa', 'historyczna', 'historyczna', 'obyczajowa'],
    'cena': [79.90, 49.00, 39.00, 42.00, 45.00, 55.00, 35.00, 52.00, 38.00, 48.00, 59.00, 44.00],
    'strony': [912, 376, 198, 295, 320, 512, 180, 263, 153, 618, 688, 432]
})

# Tabela3 zamówienia
np.random.seed(2026)
n = 80
zamowienia = pd.DataFrame({
    'zam_id': range(5001, 5001 + n),
    'ksiazka_id': np.random.choice(ksiazki['ksiazka_id'], size=n),
    'ilosc': np.random.randint(1, 6, size=n),
    'data': pd.to_datetime('2026-01-01') + pd.to_timedelta(np.random.randint(0, 120, n), unit='D'),
    'kanal': np.random.choice(['web', 'aplikacja', 'telefon'], size=n, p=[0.6, 0.3, 0.1]),
    'miasto': np.random.choice(['Warszawa', 'Kraków', 'Wrocław', 'Gdańsk', 'Poznań'], size=n)
})

In [2]:
# Krok1 tylko dopasowania
ksiazki_z_autorami = ksiazki.merge(autorzy, on='autor_id')

# Krok2 Test złączeń
ksiazki_test = pd.concat([ksiazki, pd.DataFrame({'ksiazka_id':[999], 'autor_id':[999], 'tytul':['Bez autora'], 'cena':[30.0]})], ignore_index=True)

print(f"INNER (A ∩ B): {ksiazki_test.merge(autorzy, on='autor_id', how='inner').shape[0]}")
print(f"LEFT (całe A): {ksiazki_test.merge(autorzy, on='autor_id', how='left').shape[0]}")

INNER (A ∩ B): 12
LEFT (całe A): 13


In [3]:
# Łączymy 3 tabel
pelne = zamowienia.merge(ksiazki, on='ksiazka_id').merge(autorzy, on='autor_id')

# kolumny analicztyne
pelne['wartosc'] = pelne['ilosc'] * pelne['cena']
pelne['miesiac'] = pelne['data'].dt.month
pelne['autor_pelne'] = pelne['imie'] + ' ' + pelne['nazwisko']
pelne['kat_cenowa'] = np.where(pelne['cena'] >= 50, 'droga', 'tania')

print(f"Łączny przychód: {pelne['wartosc'].sum():.2f} zł")

Łączny przychód: 11963.80 zł


In [4]:
# Zadanie 3c
raport_autorzy = pelne.groupby('autor_pelne').agg(
    liczba_zamowien=('zam_id', 'count'),
    laczna_sprzedaz=('wartosc', 'sum'),
    srednia_ilosc=('ilosc', 'mean')
).round(2).sort_values('laczna_sprzedaz', ascending=False)

print(raport_autorzy)

                     liczba_zamowien  laczna_sprzedaz  srednia_ilosc
autor_pelne                                                         
Olga Tokarczuk                    18           3389.8           2.72
Stanisław Lem                     15           2334.0           3.80
Andrzej Sapkowski                 14           1930.0           2.86
Szczepan Twardoch                 11           1580.0           2.91
Ryszard Kapuściński                5           1144.0           4.40
Jacek Dehnel                       5            748.0           3.40
Wisława Szymborska                 7            420.0           1.71
Dorota Masłowska                   5            418.0           2.20


In [5]:
# zadanie 4a/b: Przychód Kategoria vs Miesiąc
pivot = pd.pivot_table(pelne, 
                       index='kategoria', 
                       columns='miesiac', 
                       values='wartosc', 
                       aggfunc='sum', 
                       fill_value=0, 
                       margins=True, 
                       margins_name='RAZEM')

# zadanie 4c/d skąd przychodzą zamówienia?
kanaly_miasta = pd.crosstab(pelne['kanal'], pelne['miasto'], normalize='index') * 100
print(kanaly_miasta.round(1))

miasto     Gdańsk  Kraków  Poznań  Warszawa  Wrocław
kanal                                               
aplikacja    30.4     4.3    21.7      26.1     17.4
telefon       9.1    36.4     9.1      27.3     18.2
web          10.9    19.6    30.4      19.6     19.6
